# Signal précoce nouvelles cartes — views_week + text_synergy + OCG

**Problème** :  fonctionne sur les archetypes établis en OCG.  
Pour une carte qui vient de sortir (0 decks en tournoi), le signal est aveugle.

**Solution** : 3 signaux indépendants du nombre de decklists :
1.  — les joueurs regardent déjà des vidéos dessus (TOK-5)
2.  avec les cartes meta actuelles — l effet ressemble à ce qui est fort (TOK-6)
3.  de l archetype si disponible (SB-Z)

**Résultat** : table  — top cartes à acheter AVANT les decklists.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import warnings
import datetime
warnings.filterwarnings("ignore")

con = sqlite3.connect("../data/yugioh.db")

# Nouvelles cartes (TCG 2026 ou OCG depuis juin 2025)
# Note : ban_tcg IS NULL = carte légale (pas sur la banlist)
new_cards = pd.read_sql("""
    SELECT id, name, archetype, tcg_date, ocg_date, views_week, ban_tcg
    FROM cards
    WHERE (tcg_date >= '2026-01-01' OR ocg_date >= '2025-06-01')
      AND (ban_tcg IS NULL OR ban_tcg != 'Forbidden')
      AND views_week IS NOT NULL
""", con)

print(f"Nouvelles cartes : {len(new_cards):,}")
print(f"Avec archetype   : {new_cards.archetype.notna().sum()}")
print(f"Sans archetype   : {new_cards.archetype.isna().sum()}")

## Signal 1 — views_week normalisé

In [ ]:
new_cards["views_log"]    = np.log1p(new_cards["views_week"].fillna(0))
new_cards["signal_views"] = (new_cards["views_log"] / new_cards["views_log"].max()).round(4)

print("Top 15 nouvelles cartes par views_week :")
print(new_cards[["name","archetype","views_week","signal_views"]]
      .sort_values("views_week", ascending=False).head(15).to_string(index=False))

## Signal 2 — text_synergy avec les cartes meta

On identifie les archetypes meta récents (6 derniers mois via ),  
puis on calcule pour chaque nouvelle carte son score de proximité textuelle avec ces cartes.

In [ ]:
# Archetypes meta récents
meta_archs = pd.read_sql("""
    SELECT archetype FROM meta_scores
    WHERE month >= '2025-12-01'
    GROUP BY archetype ORDER BY AVG(meta_score) DESC LIMIT 20
""", con)["archetype"].tolist()

meta_cards_names = pd.read_sql(
    f"SELECT DISTINCT name FROM cards WHERE archetype IN ({{",".join(["?"]*len(meta_archs))})",
    con, params=meta_archs
)["name"].tolist()
print(f"Archetypes meta : {len(meta_archs)}, cartes de référence : {len(meta_cards_names)}")

new_set  = set(new_cards["name"])
meta_set = set(meta_cards_names)

# Synergies textuelles entre nouvelles cartes et cartes meta
ts = pd.read_sql(
    "SELECT card_a, card_b, text_synergy_score FROM text_synergies WHERE text_synergy_score >= 0.1",
    con
)
ts_nm = ts[
    (ts["card_a"].isin(new_set) & ts["card_b"].isin(meta_set)) |
    (ts["card_b"].isin(new_set) & ts["card_a"].isin(meta_set))
].copy()

ts_nm["card_new"]  = ts_nm.apply(lambda r: r["card_a"] if r["card_a"] in new_set else r["card_b"], axis=1)
ts_nm["card_meta"] = ts_nm.apply(lambda r: r["card_b"] if r["card_a"] in new_set else r["card_a"], axis=1)

ts_agg = ts_nm.groupby("card_new").agg(
    max_meta_synergy =("text_synergy_score", "max"),
    top3_meta_synergy=("text_synergy_score", lambda x: x.nlargest(3).mean()),
    n_meta_connections=("text_synergy_score", "count"),
    best_meta_match  =("card_meta", lambda x: x.iloc[x.values.argmax()])
).reset_index()
ts_agg["signal_text"] = (ts_agg["top3_meta_synergy"] / ts_agg["top3_meta_synergy"].max()).round(4)

print(f"Nouvelles cartes avec synergies meta : {len(ts_agg)}")

## Signal 3 — OCG alert score de l'archetype

In [ ]:
boutique = pd.read_sql("SELECT archetype, alert_score FROM boutique_alerts", con)
boutique["signal_ocg"] = (boutique["alert_score"] / boutique["alert_score"].max()).round(4)
boutique_map = boutique.set_index("archetype")["signal_ocg"].to_dict()
new_cards["signal_ocg"] = new_cards["archetype"].map(boutique_map).fillna(0.0)

## Fusion — Score d'alerte précoce



In [ ]:
result = new_cards.merge(
    ts_agg[["card_new","signal_text","max_meta_synergy","top3_meta_synergy",
            "n_meta_connections","best_meta_match"]],
    left_on="name", right_on="card_new", how="left"
).drop(columns="card_new")

for col in ["signal_text","max_meta_synergy","top3_meta_synergy"]:
    result[col] = result[col].fillna(0.0)
result["n_meta_connections"] = result["n_meta_connections"].fillna(0).astype(int)
result["best_meta_match"]    = result["best_meta_match"].fillna("")

result["early_score"] = (
    0.35 * result["signal_views"] +
    0.35 * result["signal_text"]  +
    0.30 * result["signal_ocg"]
).round(4)
result["early_score_100"] = (result["early_score"] / result["early_score"].max() * 100).round(1)
result = result.sort_values("early_score_100", ascending=False).reset_index(drop=True)

cols = ["name","archetype","views_week","signal_views","signal_text","signal_ocg","early_score_100","best_meta_match"]
print("=== TOP 20 SIGNAL PRÉCOCE ===")
print(result[cols].head(20).to_string(index=False))

In [ ]:
# Cartes orphelines : pas de signal OCG mais fort views + text
print("=== ORPHELINES (signal_ocg=0) FORT SIGNAL VIEWS+TEXT ===")
orphan = result[(result["signal_ocg"]==0) & (result["signal_views"]>=0.3) & (result["signal_text"]>=0.2)]
print(orphan[cols].head(10).to_string(index=False))

print()
print("=== TRIPLE SIGNAL (conviction maximale) ===")
triple = result[(result["signal_views"]>=0.4) & (result["signal_text"]>=0.3) & (result["signal_ocg"]>=0.3)]
print(triple[cols].head(10).to_string(index=False))

## Sauvegarde en base

In [ ]:
con2 = sqlite3.connect("../data/yugioh.db")
con2.execute("DROP TABLE IF EXISTS early_card_signals")
con2.execute("""
    CREATE TABLE early_card_signals (
        card_name TEXT PRIMARY KEY, archetype TEXT, tcg_date TEXT, ocg_date TEXT,
        views_week INTEGER, signal_views REAL, signal_text REAL, signal_ocg REAL,
        early_score REAL, early_score_100 REAL, max_meta_synergy REAL,
        n_meta_connections INTEGER, best_meta_match TEXT, computed_at TEXT
    )
""")
out = result[["name","archetype","tcg_date","ocg_date","views_week",
              "signal_views","signal_text","signal_ocg","early_score","early_score_100",
              "max_meta_synergy","n_meta_connections","best_meta_match"]].copy()
out.columns = ["card_name"] + list(out.columns[1:])
out["computed_at"] = datetime.date.today().isoformat()
out.to_sql("early_card_signals", con2, if_exists="append", index=False)
con2.commit()
con2.close()
print(f"Saved {len(out)} cards to early_card_signals")

## Visualisation

In [ ]:
import plotly.express as px

top10 = result.head(10).copy()
fig = px.bar(
    top10, x="early_score_100", y="name", color="archetype",
    orientation="h", text="early_score_100",
    title="Top 10 cartes — Score d'alerte précoce (views + text synergy + OCG)",
    labels={"early_score_100": "Score /100", "name": ""}, height=500,
)
fig.update_layout(yaxis={"categoryorder": "total ascending"},
                  plot_bgcolor="#1a1a2e", paper_bgcolor="#1a1a2e", font_color="white")
fig.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig.write_html("../data/early_signal_top10.html")
fig.show()

# Scatter views vs text_synergy
plot_df = result.head(50).copy()
plot_df["label"] = plot_df["name"].apply(lambda x: x[:20])
fig2 = px.scatter(
    plot_df, x="signal_views", y="signal_text",
    size="early_score_100", color="signal_ocg",
    text="label", hover_name="name",
    hover_data=["archetype","views_week","early_score_100"],
    title="Signal views vs synérgie textuelle — top 50 nouvelles cartes",
    color_continuous_scale="Reds", size_max=40, height=600,
)
fig2.update_traces(textposition="top center")
fig2.update_layout(plot_bgcolor="#1a1a2e", paper_bgcolor="#1a1a2e", font_color="white")
fig2.write_html("../data/early_signal_scatter.html")
fig2.show()
print("Graphes sauvegardés : data/early_signal_top10.html + data/early_signal_scatter.html")